# C_3: IndoBERT (Indonesia) + fitur Inggris terbaik dari C_2

Data: corpus gabungan (`load_all("id")`): artikel Inggris terfilter + artikel CNBC Indonesia. Split: sama dengan B_2.

Dua rancangan yang diuji (masing-masing dengan LR dan HGB):
- **`C3_id`**: harga + fitur IndoBERT dari artikel Indonesia saja. Ini pembanding: apakah teks Indonesia membawa sinyal sendiri?
- **`C3_en_id`**: harga + fitur Inggris terbaik C_2 (FinBERT, judul+lead per artikel, pooling CLS) + fitur IndoBERT.
  Bandingkan dengan C_2 (`C2_finbert_art_cls_*`): selisihnya adalah efek menambah teks Indonesia.

Cara kerja fitur IndoBERT:
1. Tiap artikel Indonesia menjadi satu teks: judul + `LEAD_CHARS` karakter pertama isi artikel (isi artikel diawali lead, jadi ini setara "judul + lead" di C_2).
2. Tiap artikel di-embed sendiri oleh IndoBERT (maks 510 token), lalu vektor artikel dirata-rata menjadi vektor harian.
3. PCA (dilatih di hari train yang punya artikel Indonesia saja) mengecilkan 768 angka menjadi `N_COMP`.
4. Hari tanpa artikel Indonesia (sekitar 27% hari) diberi 0 + penanda `has`.

Catatan data: kolom `lead_paragraph_gabungan_id` tidak dipakai. Pipeline Tugas 2 (`00d_alignment`, fungsi `cap_daily_text`) memotong teks lead per hari
menjadi 1.500 kata dan, pada 309 dari 881 hari Indonesia yang melewati batas itu, kata-katanya digabung dengan " | " sehingga lead per artikel tidak bisa dipisah.
Kolom judul dan konten utuh dan jumlahnya cocok dengan `jumlah_berita_id`, jadi dipakai itu.


In [1]:
from c_common import *
import json
import torch
from sklearn.decomposition import PCA

D, text = load_all("id")   # corpus gabungan: kolom *_en dan *_id
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 9)


## Pengaturan (ubah baris-baris ini saja)

In [2]:
POOL = "cls"                   # pooling IndoBERT: "cls" | "mean". Dipilih "cls" karena varian terbaik C_2 memakai CLS.
                               # Kalau diganti, itu konfigurasi tambahan: naikkan n_configs_tried di sel evaluasi.
N_COMP = 20                    # jumlah komponen PCA per sumber teks
MODEL_NAME = "indobenchmark/indobert-base-p1"
LEAD_CHARS = 1000              # karakter awal isi artikel yang dipakai (lead median 880, persentil 99 = 969 karakter)
CHUNK = 510                    # 512 dikurangi token [CLS] dan [SEP]
BATCH = 16
LIMIT = None                   # angka kecil (mis. 50) = hanya 50 hari pertama, untuk uji cepat (tidak disimpan). None = semua.

f_cls = FEATURES_C / "c3_indobert_cls.npy"
f_mean = FEATURES_C / "c3_indobert_mean.npy"

## 1. Muat IndoBERT
Pertama kali dijalankan, model diunduh (+/- 0,5 GB) ke cache Hugging Face bawaan (di drive C:, sekitar 0,5 sampai 1 GB). Jalankan sel ini hanya jika kamu siap mengunduh.

In [3]:
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()
print(MODEL_NAME, "| parameter:", round(sum(p.numel() for p in model.parameters()) / 1e6), "juta")

W1120 16:35:31.158000 27488 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

indobenchmark/indobert-base-p1 | parameter: 124 juta


## 2. Satu teks per artikel Indonesia
Judul dipisah dengan ` | ` dan isi artikel dengan `\n\n---\n\n` (dibuat di `00d_alignment`). Jumlah keduanya sudah dicek sama dengan `jumlah_berita_id` di seluruh 881 hari.

In [4]:
SEP = "\n\n---\n\n"


def build_rows(text_df, tok):
    """Satu baris per artikel Indonesia: "judul. awal isi artikel" (dipotong 510 token).
    Return list of (posisi_hari, id_artikel, token_ids)."""
    rows, art_id, fallback = [], 0, 0
    for pos, (t, c) in enumerate(zip(text_df["judul_gabungan_id"].fillna(""), text_df["konten_gabungan_id"].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        arts = [a.strip() for a in c.split(SEP) if a.strip()]
        if len(titles) != len(arts):                 # tidak cocok (mis. ada " | " di dalam judul) -> isi artikel saja
            titles, fallback = [""] * len(arts), fallback + 1
        for ti, ar in zip(titles, arts):
            ids = tok(f"{ti}. {ar[:LEAD_CHARS]}".strip(" ."), add_special_tokens=False, truncation=True, max_length=CHUNK)["input_ids"]
            rows.append((pos, art_id, ids))
            art_id += 1
    print("hari memakai isi artikel saja (jumlah judul dan isi tidak cocok):", fallback)
    return rows


has_id = (text["jumlah_berita_id"] > 0).values
print("hari dengan artikel Indonesia:", int(has_id.sum()), "dari", len(text), "| artikel:", int(text["jumlah_berita_id"].sum()))

hari dengan artikel Indonesia: 881 dari 1202 | artikel: 10075


## 3. Embed semua artikel (pekerjaan panjang, hasil disimpan)
Jalankan sekali. Kalau file `c3_indobert_*.npy` sudah ada, langsung dimuat dari sana. Di GPU beberapa menit; di CPU jauh lebih lama.

In [5]:
@torch.no_grad()
def embed_chunks(rows, tok, model, batch=BATCH):
    """IndoBERT untuk semua artikel. Return (CLS, mean) masing-masing (n_artikel, 768)."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev)
    model.to(dev).eval()
    order = np.argsort([len(r[2]) for r in rows])        # urut menurut panjang -> padding minimal
    cls = np.zeros((len(rows), 768), dtype="float32")
    mean = np.zeros_like(cls)
    for s in range(0, len(order), batch):
        ix = order[s:s + batch]
        enc = tok.pad({"input_ids": [[tok.cls_token_id] + list(rows[i][2]) + [tok.sep_token_id] for i in ix]}, return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        cls[ix] = h[:, 0].float().cpu().numpy()
        mean[ix] = ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()   # padding diabaikan
        if (s // batch) % 200 == 0:
            print(f"  {s}/{len(order)} artikel")
    model.cpu()
    return cls, mean


def pool_to_days(rows, vecs, n_days):
    """Rata-rata artikel -> hari. Hari tanpa artikel = vektor nol."""
    df = pd.DataFrame(vecs)
    df["day"] = [r[0] for r in rows]
    day_vec = df.groupby("day").mean()
    out = np.zeros((n_days, vecs.shape[1]), dtype="float32")
    out[day_vec.index.values] = day_vec.values
    return out


FEATURES_C.mkdir(parents=True, exist_ok=True)
if LIMIT is None and f_cls.exists() and f_mean.exists():
    emb = {"cls": np.load(f_cls), "mean": np.load(f_mean)}
    print("dimuat dari cache:", emb["cls"].shape)
else:
    rows = build_rows(text, tok)
    if LIMIT:
        keep = set(np.where(has_id)[0][:LIMIT])
        rows = [r for r in rows if r[0] in keep]
    print(len(rows), "artikel")
    c, m = embed_chunks(rows, tok, model)
    emb = {"cls": pool_to_days(rows, c, len(text)), "mean": pool_to_days(rows, m, len(text))}
    if LIMIT is None:                                    # jangan simpan cache kalau hanya uji sebagian
        np.save(f_cls, emb["cls"]); np.save(f_mean, emb["mean"])
        print("disimpan ke", FEATURES_C)
print("NaN:", bool(np.isnan(emb[POOL]).any()), "| hari dengan vektor:", int((np.abs(emb[POOL]).sum(1) > 0).sum()))

dimuat dari cache: (1202, 768)
NaN: False | hari dengan vektor: 881


## 4. Fitur harian: PCA (dilatih di train saja)
Fitur Inggris dibuat ulang dari embedding C_2 yang tersimpan (`c2_finbert_art_cls.npy`) dengan PCA yang sama seperti di `05b_C2_transformer`, sehingga kolomnya identik dengan varian terbaik C_2.

In [6]:
def add_pca_features(emb, prefix, has):
    """PCA di-fit HANYA pada hari train yang punya teks (vektor nol tidak ikut), lalu diterapkan ke semua hari.
    Hari tanpa teks diberi 0 pada semua komponen + penanda has. Fitur digabung ke D berdasarkan tanggal."""
    has = np.asarray(has, bool)
    feats = pd.DataFrame(emb, index=text.index)
    tr = D["train"]["date"]
    tr = tr[has[text.index.get_indexer(tr)]]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[tr].values)
    z = pca.transform(feats.values)
    z[~has] = 0.0
    comp = pd.DataFrame(z, index=text.index, columns=[f"{prefix}_{i}" for i in range(N_COMP)])
    comp[f"{prefix}_has"] = has.astype(float)
    for s in D:
        D[s] = D[s].drop(columns=[c for c in comp.columns if c in D[s].columns]) \
                   .merge(comp, left_on="date", right_index=True, how="left")
    print(prefix, "variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))
    return list(comp.columns)


# fitur IndoBERT (artikel Indonesia)
ID = add_pca_features(emb[POOL], f"c3_id_{POOL}", has_id)

# fitur Inggris terbaik C_2: FinBERT, judul+lead per artikel, CLS (dari 05b_C2_transformer)
en_cls = np.load(FEATURES_C / "c2_finbert_art_cls.npy")
has_en = (text["jumlah_berita_en"] > 0).values
assert int((np.abs(en_cls).sum(1) > 0).sum()) == int(has_en.sum()), "embedding C_2 tidak cocok dengan hari yang punya artikel Inggris"
EN = add_pca_features(en_cls, "c2_art_cls", has_en)

c3_id_cls variansi dijelaskan PCA: 0.742


c2_art_cls variansi dijelaskan PCA: 0.792


## 5. Nilai di validation
Test **tidak** disentuh. Ada 4 konfigurasi (2 rancangan × 2 model); pooling tetap `POOL`.

In [7]:
VARIANTS = {
    "C3_id_LR":     ("LR",  ID),
    "C3_id_HGB":    ("HGB", ID),
    "C3_en_id_LR":  ("LR",  EN + ID),
    "C3_en_id_HGB": ("HGB", EN + ID),
}
# n_configs_tried = 4: 2 rancangan x 2 model. Laporkan jujur (C_1: 2, C_2: 6, C_3: 4).
for name, (kind, cols) in VARIANTS.items():
    evaluate_val(name, kind, D, PRICE + cols, n_configs_tried=len(VARIANTS),
                 note=f"IndoBERT {POOL}, judul+{LEAD_CHARS} karakter per artikel, PCA {N_COMP}" + (" + FinBERT C_2" if "en_id" in name else ""))

C3_id_LR [LR] val: DA=0.500, BalAcc=0.503, F1_macro=0.497, MCC=0.007, AUC=0.484


C3_id_HGB [HGB] val: DA=0.562, BalAcc=0.561, F1_macro=0.561, MCC=0.123, AUC=0.541
C3_en_id_LR [LR] val: DA=0.483, BalAcc=0.483, F1_macro=0.483, MCC=-0.033, AUC=0.482


C3_en_id_HGB [HGB] val: DA=0.557, BalAcc=0.555, F1_macro=0.554, MCC=0.110, AUC=0.540


## 6. Bandingkan
Harga saja, C_1, C_2 (varian Inggris terbaik, `art_cls`), dan C_3. Semua skor validation; test belum disentuh.

In [8]:
rows = []
for k in ["LR", "HGB"]:
    _, sc, _ = tune(k, D, PRICE)
    rows.append({"model": f"harga saja {k}", **sc})
for name in ["C1_LR", "C1_HGB", "C2_finbert_art_cls_HGB"] + list(VARIANTS):
    f = RESULTS_C / f"{name}.json"
    if f.exists():
        rows.append({"model": name, **json.loads(f.read_text())["val"]})
    else:
        print("belum ada:", name)
pd.DataFrame(rows).round(3)

,model,DA,BalAcc,F1_macro,MCC,AUC
0,harga saja LR,0.506,0.508,0.504,0.016,0.501
1,harga saja HGB,0.523,0.521,0.520,0.042,0.503
2,C1_LR,0.500,0.501,0.500,0.001,0.441
3,C1_HGB,0.528,0.533,0.523,0.068,0.547
4,C2_finbert_art_cls_HGB,0.591,0.588,0.588,0.179,0.568
5,C3_id_LR,0.500,0.503,0.497,0.007,0.484
6,C3_id_HGB,0.562,0.561,0.561,0.123,0.541
7,C3_en_id_LR,0.483,0.483,0.483,-0.033,0.482
8,C3_en_id_HGB,0.557,0.555,0.554,0.110,0.540
